In [ ]:
from copy import deepcopy
from itertools import groupby 
import numpy as np 
import matplotlib.pyplot as plt
import neuralnet as nn
from neuralnet.preprocess import prepare_data, hot_encode
from neuralnet.activations import step, derivative_htanh , relu, derivative_relu
from neuralnet.losses import l2_hinge_loss_gradient
from neuralnet.spin_glass_utils import form_J_matrix, form_block_matrix, map_nn_to_spin_glass
from joblib import Parallel, delayed




class QuantumNeuralNetwork(nn.NeuralNetwork):
    """
    A class for constructing and training a fully connected quantum neural network.

    Attributes:
        hidden_layer_n (int): Number of hidden layers in the network.
        layer_n (int): Total number of layers (input + hidden + output).
        layer_sizes (np.array): List of sizes for each layer in the network.
        bias (list): List of bias vectors for each layer.
        weights (list): List of weight matrices connecting the layers.
        activation_f (callable): Activation function (default: sigmoid).
        activation_df (callable): Derivative of the activation function.
        cost_function (callable): Cost function for training (if provided).

    Methods:
        train(minibatch=True, minibatch_pool=10, iterations=100, η=1e-6) -> 'NeuralNetwork':
            Trains the neural network using gradient descent.

    Parameters:
        hidden_layer (int): Number of hidden layers in the network.
        layer_sizes (list[int | float]): List of hidden layer sizes (default: [10]).
        activation_function (callable): Activation function for all layers (default: sigmoid).
        activation_derivative (callable): Derivative of the activation function (default: derivative_sigmoid).
        cost (callable): Cost function to minimize during training (optional).
        cost_grad (callable): Cost function gradient with respect to activations solely

    Train Method Parameters:
        minibatch (bool): Whether to use mini-batch gradient descent (default: True).
        minibatch_pool (int | float): Number of samples per mini-batch (default: 10).
        iterations (int | float): Number of training iterations (default: 100).
        η (int | float): Learning rate for gradient descent (default: 1e-6).

    Returns:
        NeuralNetwork: The trained neural network object.

    Example:
        nn = NeuralNetwork(
                           layer_sizes=[10,64, 32,1],
                           activation_function=sigmoid,
                           activation_derivative=derivative_sigmoid)
        nn.train(minibatch=True, minibatch_pool=32, iterations=1000, η=0.01)
    """

    def __init__(
        self,
        layer_sizes: (list[int] | list[float]),
        activation_function: callable,
        activation_derivative: callable,
        cost_function: callable,
        cost_grad: callable,
        quantumness: (float | int)
    ) -> 'QuantumNeuralNetwork':
        
        super().__init__(layer_sizes,
                          activation_function,
                          activation_derivative,
                          cost_function,
                          cost_grad)

        self.weights = 
        self.quantumness = quantumness

    def __str__(self):
        """
        Returns a string representation of the neural network's architecture, weights, and biases.
        """
        display_str = "Neural Network Structure:\n"
        display_str += f"Number of Layers: {self.layer_n}\n"
        display_str += f"Hidden Layers: {self.hidden_layer_n}\n"
        display_str += "Layer Sizes: " + " -> ".join(map(str, self.layer_sizes)) + "\n\n"

        display_str += "Biases:\n"
        for i, bias in enumerate(self.bias, start=1):
            display_str += f"  Layer {i + 1}: Shape {bias.shape}\n"

        display_str += "\nWeights:\n"
        for i, weight in enumerate(self.weights, start=1):
            display_str += f"  Layer {i}: Shape {weight.shape}\n"

        display_str += f"\nActivation Function: {self.activation_f.__name__}\n"
        display_str += f"Activation Derivative: {self.activation_df.__name__}\n"
        display_str += f"Cost Function: {self.cost.__name__}\n"
        display_str += f"Cost Gradient: {self.cost_grad.__name__}\n"

        return display_str
    
    def feed(ψ:np.ndarray[:,:], ) -> np.ndarray[:]:

        activations = [ψ]
        for l in range(1, self.layer_n - 1, 1):
            measure( )
            rotate()



    def train(
        self,
        input,
        output,
        momentum: (int | float) = None,
        minibatch: bool = True,
        minibatch_pool: (int | float) = 10,
        iterations: (int | float) = 100,
        η: (int | float) = 1e-6,
    ) -> None:
        """
        Trains the neural network using gradient descent.

        Parameters:
            minibatch (bool): Whether to use mini-batch gradient descent (default: True).
            minibatch_pool (int | float): Size of the mini-batch for training (default: 10).
            iterations (int | float): Number of training iterations (default: 100).
            η (int | float): Learning rate for gradient descent (default: 1e-6).

        Returns:
            NeuralNetwork: The trained neural network object.

        Description:
            - Implements forward propagation for each input to compute activations.
            - Performs backpropagation to compute gradients for weights and biases.
            - Updates weights and biases using gradient descent.
            - Supports mini-batch gradient descent if `minibatch` is set to True.

        Example:
            nn.train(minibatch=True, minibatch_pool=32, iterations=1000, η=0.01)
        """

        for _ in range(iterations):
            if minibatch:
                indexes = np.random.choice(input.shape[0], size=minibatch_pool)
                X, Y = input[indexes], output[indexes]
            else:
                X, Y = input, output

            w_grads = [np.zeros(matrix.shape) for matrix in self.weights]

            b_grads = [np.zeros(vector.shape) for vector in self.bias]

            if momentum is not None:
                v = [np.zeros(w.shape) for w in self.weights]

            # iterate for each set of x and y
            # find zs and as (pre-act and activation)
            for x, y in zip(X, Y):
                # print("x id:",id(x))

                # def "quantum" feedforward
                z0 = self.weights[0] @ x + self.bias[0]
                zs = [z0]
                a0 = self.activation_f(z0)
                activations = [a0]
                for l in range(1, self.layer_n - 1, 1):
                    # print("layers:",l,l-1)
                    zl = self.weights[l] @ activations[l - 1] + self.bias[l]
                    activation = self.activation_f(zl)
                    # print("activation:", activation)
                    zs.append(zl)
                    activations.append(activation)

                z_output = zs[-1]
                a_output = activations[-1]
                output_error = self.cost_grad(a_output, y) * self.activation_df(
                    z_output
                )
                errors = [output_error]
                for l in range(self.hidden_layer_n, 0, -1):
                    error = (
                        self.weights[l].T @ errors[-1] * self.activation_df(zs[l - 1])
                    )
                    errors.append(error)

                errors.reverse()
                # compute sum of error
                for l in range(0, self.hidden_layer_n + 1, 1):
                    # w_grads[l] += errors[l]@activations[l].T
                    w_grads[l] += np.outer(
                        errors[l], activations[l - 1] if l > 0 else x
                    )
                    # print(w_grads)
                    b_grads[l] += errors[l]
                    # print(b_grads)

            # gradient descent
            if momentum is None:
                for l in range(0, self.hidden_layer_n + 1):
                    self.weights[l] -= η / minibatch_pool * w_grads[l]
                    self.bias[l] -= η / minibatch_pool * b_grads[l]
            else:
                γ = momentum
                for l in range(0, self.hidden_layer_n + 1):
                    v[l] = γ * v[l] + η / minibatch_pool * w_grads[l]
                    self.weights[l] -= v[l]
                    self.bias[l] -= η / minibatch_pool * b_grads[l]

    def predict(self, input: list[np.ndarray]) -> list[np.ndarray]:
        """
        Predicts the output for a given input using the trained neural network.

        Parameters:
            input (np.array): Input data to predict, where each row corresponds to a single input instance.

        Returns:
            list: A list of predictions where each prediction corresponds to the output of the neural network
                  for the corresponding input instance.

        Description:
            - Performs forward propagation through the network to compute the output layer activations.
            - Returns the final layer activations as predictions.

        Example:
            predictions = nn.predict(x_test)
        """

        results = []
        for x in input:
            # print(x,"\n")
            z0 = self.activation_f(self.weights[0] @ x + self.bias[0])
            activations = [z0]
            for l in range(1, self.layer_n - 1):
                zl = self.weights[l] @ activations[l - 1] + self.bias[l]
                a = self.activation_f(zl)
                activations.append(a)

            results.append(activations[-1])

        return results



NameError: name 'NEural' is not defined